# K-Means and Comparing Clustering Methods

**Series:** Spatial Clustering (notebook 3 of 3)
**Time:** about 30 minutes

**In this notebook you will:**
- Split points into a fixed number of groups with K-Means
- Choose the number of groups with the elbow method
- Try hierarchical clustering
- Compare K-Means with DBSCAN and decide which fits your question

**You should already know:** notebooks 03a and 03b.

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## How K-Means works

You choose **k**, the number of clusters. K-Means places k centers, assigns every point to the
nearest center, moves each center to the middle of its points, and repeats until nothing changes.

Unlike DBSCAN, **every point is assigned to a cluster** (there is no noise), and clusters tend to be
round and similar in size. That makes K-Means good for dividing an area into zones, and weaker at
finding true hotspots.

## Step 1: Load and prepare the data

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

# Read the CSV straight from GitHub into memory (about 40 MB, nothing is saved to disk)
crimes = pd.read_csv("https://raw.githubusercontent.com/SpatialTurn/DataAnalysis/main/data/CrimesChicago20220225.csv")
crimes.columns = crimes.columns.str.strip()          # some column names start with a space
crimes = crimes.dropna(subset=["LATITUDE", "LONGITUDE"])

In [ ]:
# Keep three crime types and turn them into points measured in meters
crime_types = ["WEAPONS VIOLATION", "ROBBERY", "NARCOTICS"]
subset = crimes[crimes["PRIMARY DESCRIPTION"].isin(crime_types)]

points = gpd.GeoDataFrame(
    subset,
    geometry=gpd.points_from_xy(subset["LONGITUDE"], subset["LATITUDE"]),
    crs="EPSG:4326",
).to_crs(epsg=32616)      # UTM zone 16N: x and y are in meters

robbery = points[points["PRIMARY DESCRIPTION"] == "ROBBERY"].copy()
coords = np.column_stack([robbery.geometry.x, robbery.geometry.y])
print(len(robbery), "robberies")

## Step 2: Choose k with the elbow method

**Inertia** is the total squared distance from points to their center. It always drops as k grows.
We look for the "elbow" where adding more clusters stops helping much.

In [ ]:
from sklearn.cluster import KMeans

ks = range(2, 13)
inertia = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(coords).inertia_ for k in ks]

plt.plot(list(ks), inertia, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow method")
plt.show()

## Step 3: Run K-Means with k = 6

In [ ]:
km = KMeans(n_clusters=6, n_init=10, random_state=42).fit(coords)
robbery["kmeans"] = km.labels_

ax = robbery.plot(column="kmeans", cmap="tab10", markersize=1, categorical=True,
                  legend=True, figsize=(7, 9))
centers = gpd.GeoSeries(gpd.points_from_xy(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1]),
                        crs=robbery.crs)
centers.plot(ax=ax, color="black", marker="x", markersize=100)
ax.set_title("Robbery: K-Means (k = 6), black X = cluster center")
ax.set_axis_off()
plt.show()

## Step 4: Hierarchical clustering

Hierarchical (agglomerative) clustering starts with every point as its own group and keeps merging
the two closest groups. The **dendrogram** shows the order of merges; cutting it at a height gives
your clusters. It is slow on large data, so we use a random sample of 2,000 robberies.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

sample = robbery.sample(2000, random_state=1)
sample_xy = np.column_stack([sample.geometry.x, sample.geometry.y])

tree = linkage(sample_xy, method="ward")
plt.figure(figsize=(10, 4))
dendrogram(tree, truncate_mode="lastp", p=20, no_labels=True)
plt.title("Dendrogram (last 20 merges)")
plt.ylabel("Merge distance")
plt.show()

sample["hier"] = fcluster(tree, t=6, criterion="maxclust")
sample.plot(column="hier", cmap="tab10", markersize=3, categorical=True, figsize=(7, 9))
plt.title("Hierarchical clustering, 6 clusters (2,000 point sample)")
plt.axis("off")
plt.show()

## Step 5: Compare the methods side by side

In [ ]:
from sklearn.cluster import DBSCAN
robbery["dbscan"] = DBSCAN(eps=1000, min_samples=50).fit(coords).labels_

fig, axes = plt.subplots(1, 2, figsize=(12, 8))
robbery.plot(ax=axes[0], column="kmeans", cmap="tab10", markersize=1, categorical=True)
axes[0].set_title("K-Means: every point in a zone")
noise = robbery["dbscan"] == -1
robbery[noise].plot(ax=axes[1], color="lightgray", markersize=1)
robbery[~noise].plot(ax=axes[1], column="dbscan", cmap="tab20", markersize=1, categorical=True)
axes[1].set_title("DBSCAN: dense hotspots, gray = noise")
for ax in axes:
    ax.set_axis_off()
plt.show()

## Which method should I use?

| Method | You choose | Noise? | Cluster shape | Good for |
|---|---|---|---|---|
| **K-Means** | number of clusters | No | Round, similar size | Dividing an area into zones |
| **Hierarchical** | number of clusters or a cut height | No | Flexible | Seeing structure at several scales (small data) |
| **DBSCAN** | `eps` and `min_samples` | Yes | Any shape | Hotspots when you know a meaningful distance |
| **HDBSCAN** | `min_cluster_size` | Yes | Any shape | Hotspots with varying density |

## Try it yourself

Run K-Means on **weapons violations** with the k you get from an elbow plot. Are the centers in similar places to the robbery centers?

<details>
<summary><b>Show a solution</b></summary>

```python
weapons = points[points["PRIMARY DESCRIPTION"] == "WEAPONS VIOLATION"].copy()
wc = np.column_stack([weapons.geometry.x, weapons.geometry.y])
inertia = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(wc).inertia_ for k in ks]
plt.plot(list(ks), inertia, marker="o"); plt.show()

weapons["kmeans"] = KMeans(n_clusters=6, n_init=10, random_state=42).fit_predict(wc)
weapons.plot(column="kmeans", cmap="tab10", markersize=1, categorical=True)
plt.show()
```

</details>

In [ ]:
# Write your answer here

## Recap

- K-Means splits all points into k round groups; choose k with the elbow method.
- Hierarchical clustering shows how groups merge, through a dendrogram.
- DBSCAN and HDBSCAN find dense hotspots and leave scattered points as noise.
- Pick the method that matches your question: zones or hotspots.